<a href="https://colab.research.google.com/github/maggoatt/Grounded-Text-Summarization-of-Research-Papers/blob/main/Summarization_Model_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Summarization Model Workflow

- Baseline: TextRank
- Advanced: Facebook BART (Large-CNN)
High-level pipeline:
1. Take in the selected paper (i.e. from ```streamlit``` file)
2. Sliding window (i.e. 1k tokens) to chunk paper, take note of the section titles per chunk
3. Generate summaries per chunk per model and stitch together

### Citations/references:

1. Workflow to implement TextRank:

Adapted from: ERRAJI, Yassine (June 19 2025). ["Understanding TextRank: A Deep Dive into Graph-Based Text Summarization and Keyword Extraction"](https://medium.com/@yassineerraji/understanding-textrank-a-deep-dive-into-graph-based-text-summarization-and-keyword-extraction-905d1fb5d266).
Medium Article.

2. Workflow to implement Facebook BART:

Adapted from: Lewis, Mike _et al._ (Accessed February 2026). ["BART: Denoising Sequence-to-Sequence Pre-training for Natural Language Generation, Translation, and Comprehension"](https://huggingface.co/facebook/bart-large-cnn).
Hugging Face Documentation.

Adapted from: baksapeter (April 11, 2025). ["Maximum number of input tokens"](https://huggingface.co/facebook/bart-large-cnn/discussions/83). Hugging Face Discussion.

3. Misc. syntax: scikit-learn documentation



In [1]:
# installing dependencies

%pip install scikit-learn networkx transformers # for TextRank (networkx) and BART (transformers)
%pip install textstat language-tool-python rouge-score matplotlib

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
# imports

# TextRank
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import networkx as nx
import json

# BART
import torch
from transformers import AutoTokenizer, BartForConditionalGeneration

# Baselines
import textstat
import language_tool_python
from rouge_score import rouge_scorer
import math
import numpy as np
from transformers import GPT2LMHeadModel, GPT2TokenizerFast, AutoModelForSequenceClassification, AutoTokenizer as NLITokenizer

# Evals
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import os
from nltk.tokenize import sent_tokenize

/Users/maggiezhang/Desktop/WINTER '26/CS175/Grounded-Text-Summarization-of-Research-Papers/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## TextRank Pipeline:
1. Extract + concatenate text from selected paper (to be referenced from JSON object created by UI/API request)
2. Tokenize extracted + concatenated text
3. Create similarity graph of tokens
4. Run PageRank
5. Rank by top-k and output final summary

Additionally, preserve which section the sentence originated from (for later analysis/retrieval purposes).

In [ ]:
# extracting content from paper
test_path = "../data/253098895.json"
k = 5 # summary sentence length

with open(test_path, 'r', encoding='utf-8') as f:
    paper = json.load(f) # load the selected paper's json file

cid = paper["corpusid"]
body_text = []
section_map = {} # preserving sentences' og section

# current method: concatenate all paragraphs from just the body section together. no splitting by section
for section in paper["sections"]: # (1) extract and concatenate text from selected paper
    section_title = section["section_title"]
    sentences = sent_tokenize(section["text"])

    for sentence in sentences:
        if sentence:
            section_map[len(body_text)] = section_title  # track section of sentence based on index of sentence
            body_text.append(sentence)

# body_text: all sentences
# section_map: index of sentence: section title

In [24]:
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(body_text) # (2) grab doc-term mtx, treating each sentence as a document in body_text corpus
similarity_mtx = cosine_similarity(X) # (3) cosine sim on sentences based on word importance
graph = nx.from_numpy_array(similarity_mtx)

scores = nx.pagerank(graph) # (4) score sentences via PageRank

ranked = sorted(((scores[i], s, section_map[i]) for i, s in enumerate(body_text)), reverse=True) # sentences and section name ranked by highest scores

summary = " ".join([s for _, s, _ in ranked[:k]])
print(summary) # (5)

Specifically, SPABERT linearizes the 2-dimensional spatial context by forming pseudo sentences that consist of names of the pivot and neighboring geo-entities, ordered by their spatial distance to the pivot. 1 is constructed as: The pseudo sentence starts with the pivot name followed by the names of the pivot's neighboring geo-entities, ordered by their spatial distance to the pivot in ascending order. One task is the masked language modeling (MLM) (Devlin et al., 2019), for which SPABERT needs to learn how to complete the full names of geo-entities from pseudo sentences with randomly masked subtokens using the remaining subtokens and their spatial coordinates (i.e., partial names and spatial relations between subtokens). For a query set Q = {(q i , SC(q i ))} |Q| i=1 , the goal is to find the corresponding geo-entity for each

Here, Q and C are the USGS and Wikidata geo-entities ( §3.1). SPABERT also encodes the spatial relations between the pivot and neighboring geo-entities with a c

In [ ]:
# export summary to .txt document for benchmarking analysis:
import os
os.makedirs("../summaries", exist_ok=True)

file_path = f"../summaries/{cid}_textrank_summary.txt"

with open(file_path, 'w', encoding='utf-8') as file:
    file.write(summary)

## Facebook BART Pipeline:
1. Create summarization pipeline, specifying Facebook BART (large-CNN model)
2. Extract + concatenate text from selected paper
3. Check if token count exceeds Facebook BART max input token count (1024)
4. If token count > 1024, implement sliding window. Else, summarize entire input
5. Output the final summary

In [ ]:
model_name = "facebook/bart-large-cnn" # (1)

full_body_text = ". ".join(body_text) # (2) turn the list of sentences into string

tokenizer = AutoTokenizer.from_pretrained(model_name)
bart_model = BartForConditionalGeneration.from_pretrained(model_name)
max_token_count = 1024 # BART's actual positional encoding limit

def summarize(text, max_new_tokens=300, min_new_tokens=30):
    """Summarize a single chunk of text using BART (input auto-truncated to 1024 tokens)."""
    inputs = tokenizer(text, return_tensors="pt", max_length=max_token_count, truncation=True)
    summary_ids = bart_model.generate(
        inputs["input_ids"],
        max_new_tokens=max_new_tokens,
        min_new_tokens=min_new_tokens,
        num_beams=4,
        length_penalty=2.0,
        forced_bos_token_id=0
    )
    return tokenizer.decode(summary_ids[0], skip_special_tokens=True)

def get_token_count(text):
    return len(tokenizer.encode(text, truncation=False))

def reduce_summaries(texts, round_num=1):
    """
    Recursively summarize until the combined text fits within 1024 tokens.

    1. Summarize each chunk individually
    2. Concatenate the summaries
    3. If still > 1024 tokens, group into chunks and repeat
    4. Once <= 1024 tokens, produce the final summary
    """
    print(f"--- Round {round_num}: summarizing {len(texts)} chunks ---")

    chunk_summaries = []
    for i, text in enumerate(texts):
        tc = get_token_count(text)
        summary = summarize(text)
        print(f"  chunk {i+1}/{len(texts)}: {tc} tokens -> {get_token_count(summary)} tokens")
        chunk_summaries.append(summary)

    # combine all summaries into one text
    combined = " ".join(chunk_summaries)
    combined_tokens = get_token_count(combined)
    print(f"  combined result: {combined_tokens} tokens")

    if combined_tokens <= max_token_count:
        # fits within limit — concatenate and return as-is
        print(f"  fits within {max_token_count} tokens, concatenating summaries")
        return combined
    else:
        # still too long — group summaries into 1024-token chunks and recurse
        print(f"  still > {max_token_count} tokens, splitting again...\n")
        groups = []
        current_group = []
        current_tokens = 0
        for s in chunk_summaries:
            s_tokens = get_token_count(s)
            if current_tokens + s_tokens > max_token_count and current_group:
                groups.append(" ".join(current_group))
                current_group = [s]
                current_tokens = s_tokens
            else:
                current_group.append(s)
                current_tokens += s_tokens
        if current_group:
            groups.append(" ".join(current_group))

        return reduce_summaries(groups, round_num + 1)

# --- run the pipeline ---
token_count = get_token_count(full_body_text)
print(f"total tokens: {token_count}\nmax allowed tokens: {max_token_count}\n")

if token_count > max_token_count:
    # step 1: summarize each section individually (preserves 1:1 mapping with section titles)
    section_texts = [section["text"] for section in paper["sections"]]
    summaries = []
    print(f"--- Summarizing {len(section_texts)} sections ---")
    for i, text in enumerate(section_texts):
        tc = get_token_count(text)
        s = summarize(text)
        print(f"  section {i+1}/{len(section_texts)}: {tc} tokens -> {get_token_count(s)} tokens")
        summaries.append(s)

    # step 2: combine section summaries and reduce until it fits in 1024 tokens
    combined = " ".join(summaries)
    combined_tokens = get_token_count(combined)
    print(f"\ncombined section summaries: {combined_tokens} tokens")

    if combined_tokens <= max_token_count:
        # already fits — concatenate and use as final summary
        print(f"  fits within {max_token_count} tokens, summarize once more then return")
        summary_text = summarize(combined)
    else:
        # need more reduction rounds
        print(f"  still > {max_token_count} tokens, entering reduction loop...\n")
        groups = []
        current_group = []
        current_tokens = 0
        for s in summaries:
            s_tokens = get_token_count(s)
            if current_tokens + s_tokens > max_token_count and current_group:
                groups.append(" ".join(current_group))
                current_group = [s]
                current_tokens = s_tokens
            else:
                current_group.append(s)
                current_tokens += s_tokens
        if current_group:
            groups.append(" ".join(current_group))

        summary_text = reduce_summaries(groups, round_num=2)
else:
    # small enough to summarize directly
    summary_text = summarize(full_body_text)
    summaries = [summary_text]

print(f"\n{'='*80}")
print("FINAL SUMMARY:")
print(f"{'='*80}")
print(summary_text)

In [ ]:
# export summary to .txt document for benchmarking analysis:
file_path = f"../summaries/{cid}_bart_summary.txt"

with open(file_path, 'w', encoding='utf-8') as file:
    file.write(summary_text)

## Benchmarking Analysis

- For grammar, readability, and clarity:
  - [Textstat](https://textstat.org/) - Readability scores (Flesch-Kincaid, SMOG, etc.)
  - [LanguageTool API](https://languagetool.org/http-api/) - Grammar and style checking
  - [Perplexity (Hugging Face)](https://huggingface.co/docs/transformers/en/perplexity) - Fluency proxy via GPT-2


### 1. Readability Scores (Textstat)

Measures how easy the summary is to read. Lower grade levels = more accessible.

- Flesch Reading Ease: 0-100 score (higher = easier)
- Flesch-Kincaid Grade: US grade level needed to understand
- Gunning Fog Index: Years of education needed
- SMOG Index: Years of education needed (based on polysyllables)
- Dale-Chall Score: Adjusted grade level using familiar word list

In [ ]:
def readability_scores(text, model_type):
    # computing readability metrics using textstat
    scores = {
        "Flesch Reading Ease": textstat.flesch_reading_ease(text),
        "Flesch-Kincaid Grade": textstat.flesch_kincaid_grade(text),
        "Gunning Fog Index": textstat.gunning_fog(text),
        "SMOG Index": textstat.smog_index(text),
        "Dale-Chall Score": textstat.dale_chall_readability_score(text),
    }
    print(f"{model_type} Readability:")
    for metric, val in scores.items():
        print(f"{metric}: {val:.2f}")
    return scores

textrank_readability = readability_scores(summary, "TextRank")
bart_readability = readability_scores(summary_text, "BART")

### 2. Grammar & Style (LanguageTool)

Counts grammar errors, style issues, and typos. Fewer errors = better quality.

In [ ]:
tool = language_tool_python.LanguageTool('en-US')

def grammar_check(text, model_type):
    """Count grammar/style errors using LanguageTool."""
    matches = tool.check(text)
    word_count = len(text.split())

    # categorize errors
    categories = {}
    for m in matches:
        cat = m.category
        categories[cat] = categories.get(cat, 0) + 1

    error_rate = len(matches) / word_count if word_count > 0 else 0

    print(f"{model_type} Grammar:")
    print(f"Total errors: {len(matches)}")
    print(f"Word count: {word_count}")
    print(f"Error rate: {error_rate:.2f} errors/word")
    if categories:
        print(f"By category:")
        for cat, count in sorted(categories.items(), key=lambda x: -x[1]):
            print(f"{cat}: {count}")

    # show first few errors as examples
    for m in matches[:3]:
        print(f"  >> \"{m.context}\" — {m.message}")

    return {"total_errors": len(matches), "error_rate": error_rate, "categories": categories}

textrank_grammar = grammar_check(summary, "TextRank")
bart_grammar = grammar_check(summary_text, "BART")

### 3. Perplexity (GPT-2)

Measures fluency — how "natural" the text sounds to a language model. Lower perplexity = more fluent.


In [ ]:
# perplexity using GPT-2 with a sliding window for texts longer than the model's context
# lower perplexity means more fluent/natural text

gpt2_model_name = "gpt2"
gpt2_tokenizer = GPT2TokenizerFast.from_pretrained(gpt2_model_name)
gpt2_model = GPT2LMHeadModel.from_pretrained(gpt2_model_name)
gpt2_model.eval()

def compute_perplexity(text, model_type):
    encodings = gpt2_tokenizer(text, return_tensors="pt")
    max_length = gpt2_model.config.n_positions  # 1024 for GPT-2
    stride = 512
    seq_len = encodings.input_ids.size(1)

    nlls = []
    prev_end = 0
    for begin in range(0, seq_len, stride):
        end = min(begin + max_length, seq_len)
        target_len = end - prev_end  # tokens actually score in this window

        input_ids = encodings.input_ids[:, begin:end]
        target_ids = input_ids.clone()
        # mask out tokens already score (overlap)
        target_ids[:, :-target_len] = -100

        with torch.no_grad():
            outputs = gpt2_model(input_ids, labels=target_ids)
            neg_log_likelihood = outputs.loss

        nlls.append(neg_log_likelihood * target_len)
        prev_end = end
        if end == seq_len:
            break

    ppl = torch.exp(torch.stack(nlls).sum() / end).item()

    print(f"{model_type} Perplexity:")
    print(f"Perplexity: {ppl:.2f}")
    print(f"(tokens evaluated: {seq_len})")
    return ppl

textrank_ppl = compute_perplexity(summary, "TextRank")
bart_ppl = compute_perplexity(summary_text, "BART")

### 4. Factual Consistency: NLI (DeBERTa)

Loop through each sentence in summary: check if entailed (supported), neutral (unverifiable), or contradict (hallucinated) per sentence from retrieved evidence. 

files as `{corpus_id}_{retrieval_model}_retrieval_{sentence_number}.json`

Hallucination rate = contradictions / total sentences (neutral is not counted as hallucination).

In [ ]:
# getting hte diff. summaries 
with open(f"../summaries/{cid}_bart_summary.txt", 'r') as f:
    bartsummary = f.read()

with open(f"../summaries/{cid}_textrank_summary.txt", 'r') as f:
    trsummary = f.read()

In [22]:
# model + labels
nli_model_name = "cross-encoder/nli-deberta-v3-base"
nli_tokenizer = NLITokenizer.from_pretrained(nli_model_name)
nli_model = AutoModelForSequenceClassification.from_pretrained(nli_model_name)
nli_model.eval()

NLI_LABELS = ["contradiction", "entailment", "neutral"]

Loading weights: 100%|██████████| 202/202 [00:00<00:00, 987.33it/s, Materializing param=pooler.dense.weight]                                        
DebertaV2ForSequenceClassification LOAD REPORT from: cross-encoder/nli-deberta-v3-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
deberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [7]:
def nli_score(premise, hypothesis):
    # helper func to score a single (premise, hypothesis) pair, return dict of label : probability
    inputs = nli_tokenizer(premise, hypothesis, return_tensors="pt", truncation=True, max_length=512)

    with torch.no_grad():
        logits = nli_model(**inputs).logits
    probs = torch.softmax(logits, dim=-1)[0].tolist() # get the probs from softmaxing logits

    return {label: prob for label, prob in zip(NLI_LABELS, probs)}

In [ ]:
# textrank nli (checking reliability of nli), use section_map

trsentences = sent_tokenize(trsummary)
results = []

for i, sent in enumerate(trsentences):
    print(sent)
    best_entailment = 0.0
    best_section = ""
    best_score = ""

    worst = {}
    worst_results = ""

    for ind, title in section_map.items(): # going thru each sentence to find NLI factual consistency
        p_sent = body_text[ind]
        scores = nli_score(p_sent, sent)

        if scores["entailment"] > best_entailment:
            best_entailment = scores["entailment"]
            best_section = title
            best_score = p_sent
        elif scores["contradiction"] > .50: # greater than half probability of contradict! save for analysis
            worst[p_sent] = (title, scores["contradiction"])

    if len(worst.items()) == 0: # none found (this is for formatting)
        worst_results = "None"
    else:
        for w_sent, ts_tup in worst.items():
            worst_results += f"Title: {ts_tup[0]}\nSentence: {w_sent}\nScore: {ts_tup[1]}\n\n"
    
    result = f"TEXTRANK x NLI RESULTS\n\nSentence: {sent}\n\nBest Entailment: {best_entailment}\nBest Section: {best_section}\nBest Sentence: {best_score}\n\nWorst: {worst_results}"
    
    file_path = f"../nli_benchmarks/{cid}_textrank_results_{i+1}.txt"

    with open(file_path, 'w', encoding='utf-8') as file:
        file.write(result) # save the results

Specifically, SPABERT linearizes the 2-dimensional spatial context by forming pseudo sentences that consist of names of the pivot and neighboring geo-entities, ordered by their spatial distance to the pivot.
1 is constructed as: The pseudo sentence starts with the pivot name followed by the names of the pivot's neighboring geo-entities, ordered by their spatial distance to the pivot in ascending order.
One task is the masked language modeling (MLM) (Devlin et al., 2019), for which SPABERT needs to learn how to complete the full names of geo-entities from pseudo sentences with randomly masked subtokens using the remaining subtokens and their spatial coordinates (i.e., partial names and spatial relations between subtokens).
For a query set Q = {(q i , SC(q i ))} |Q| i=1 , the goal is to find the corresponding geo-entity for each

Here, Q and C are the USGS and Wikidata geo-entities ( §3.1).
SPABERT also encodes the spatial relations between the pivot and neighboring geo-entities with a c

In [ ]:
# preparing matched retrieval sentences
retrieval_path = f"../retrievals/{cid}_faiss_retrieval"
bart_sent_sum = sent_tokenize(bartsummary)
evidence_pairs = []

for i, sent in enumerate(bart_sent_sum):

    with open(f"{retrieval_path}_{i+1}.json", "r") as f:
        retrieved = json.load(f)
        print(retrieved)

        for sec_title, sents in retrieved.items():
            for sent in sents:
                evidence_pairs.append((sec_title, sent)) # now have nice list of section title, sentence per retrieved sentences 

{'Conclusion': ['This paper presented SPABERT ( ), a language model trained on geographic datasets for contextualizing geo-entities.'], 'Introduction': ['SPABERT is a LM built upon a pretrained BERT and further trained to produce contextualized geo-entity representations given large geographic datasets.', 'To tackle these challenges, we present SPABERT ( ), a LM that captures the spatially varying semantics of geo-entity names using large geographic datasets for entity representation.']}
{'Preliminary': ['Linearizing Neighboring Geo-entity Names For a pivot, p, SPABERT first linearizes its neighboring geo-entitie names to form a BERT-compatible input sequence, called a pseudo sentence.'], 'Contextualizing Geo-entities': ['Also, SPABERT incorporates a spatial coordinate embedding mechanism, which seeks to represent the spatial relations between the pivot and its neighboring geo-entities.', "Then SPABERT averages the pivot's token-level embeddings to produce a fixedlength embedding for t

In [ ]:
# bart nli on retrieved pairs!

bartsentences = sent_tokenize(bartsummary)
results = []

for i, sent in enumerate(bartsentences):
    print(sent)
    
    file_path = f"../nli_benchmarks/{cid}_bart_results_{i+1}.txt"

    with open(file_path, 'w', encoding='utf-8') as file:
        file.write(f"BART x NLI RESULTS\n\nSentence: {sent}\n\n")

    for sec_title, r_sent in evidence_pairs: # going thru each sentence to find NLI factual consistency
        scores = nli_score(r_sent, sent)
    
        result = f"Section Title: {sec_title}\nRetrieved Sentence: {r_sent}\nEntailment Score: {scores["entailment"]}\nNeutral Score: {scores["neutral"]}\nContradiction Score: {scores["contradiction"]}\n\n"
        
        with open(file_path, 'a', encoding='utf-8') as file:
            file.write(result) # save the results

SPABERT is a LM built upon a pretrained BERT and trained to produce contextualized geo-entity representations given large geographic datasets.
For a pivot, p, SPAberT first linearizes its neighboring geo-entitie names to form a BERT-compatible input sequence, called a pseudo sentence.
The representations support various downstream applications, including contextualized Geo-entity classification.
SPABERT performs the best for the 1:125K maps (30-CA) (Tab.
4) Our hypothesis is that the test maps contain denser geo-entities than other scales.
We simulate omission using the OSM dataset by gradually removing random neighbors of a pivot entity.


### 5. ROUGE Scores

Measures n-gram overlap between the generated summary and the paper's abstract (as a reference summary).

- ROUGE-1: Unigram overlap
- ROUGE-2: Bigram overlap
- ROUGE-L: Longest common subsequence

In [ ]:
reference = None
title = paper["title"]

for section in paper["sections"]:
    if section["section_title"].lower() == "introduction":
        reference = section["text"]

if not reference:
    print("No abstract found in paper — skipping ROUGE.")
else:
    scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)

    def compute_rouge(generated, reference_text, model_type):
        scores = scorer.score(reference_text, generated)
        print(f"{model_type} ROUGE (vs. abstract):")
        for metric, vals in scores.items():
            print(f"{metric}: precision={vals.precision:.2f}  recall={vals.recall:.2f}  f1={vals.fmeasure:.2f}")
        return scores

    print(f"rouge1: 1-gram\nrouge2: bigram\nrougeL: longest common subsequence.")
    textrank_rouge = compute_rouge(trsummary, reference, "TextRank")
    bart_rouge = compute_rouge(bartsummary, reference, "BART")

In [ ]:
# aggregate metrics from all papers
metrics_dir = Path("../metrics")

all_rows = []
for tsv_path in metrics_dir.glob("*_metrics.tsv"):
    corpus_id = tsv_path.stem.replace("_metrics", "")
    df = pd.read_csv(tsv_path, sep="\t")
    df["corpusid"] = corpus_id
    all_rows.append(df)

if not all_rows:
    print("No metrics files found")
else:
    metrics_df = pd.concat(all_rows, ignore_index=True)

    # average each summarization statistic across papers, per model
    avg_df = metrics_df.groupby("model", as_index=False).mean(numeric_only=True)
    display(avg_df)

    # 1) average readability metrics by model
    readability_cols = [
        "flesch_reading_ease",
        "flesch_kincaid_grade",
        "gunning_fog_index",
        "smog_index",
        "dale_chall_score",
    ]

    # meanings of each readability metric (for interpreting the chart)
    print("Readability metrics (averaged across papers):")
    print("- Flesch Reading Ease: 0–100, higher = easier to read.")
    print("- Flesch-Kincaid Grade: approximate U.S. school grade level needed.")
    print("- Gunning Fog Index: estimated years of formal education needed.")
    print("- SMOG Index: years of education needed based on complex words.")
    print("- Dale-Chall Score: grade level using a list of familiar words.")

    fig, ax = plt.subplots(figsize=(10, 5))
    avg_df.set_index("model")[readability_cols].plot(kind="bar", ax=ax)
    ax.set_title("Average readability metrics by model (across papers)")
    ax.set_ylabel("Score")
    ax.legend(title="Metric")
    plt.tight_layout()
    plt.show()

    # 2) average grammar error rate and perplexity by model
    fig, ax = plt.subplots(figsize=(8, 5))
    subset = avg_df.set_index("model")[["error_rate", "perplexity"]]
    subset.plot(kind="bar", ax=ax)
    ax.set_title("Average grammar error rate and perplexity by model (across papers)")
    ax.set_ylabel("Value")
    ax.legend(title="Metric")
    plt.tight_layout()
    plt.show()

    # 3) example scatter plot comparing stats to each other at the paper level
    #    (readability vs fluency)
    fig, ax = plt.subplots(figsize=(6, 5))
    for model, group in metrics_df.groupby("model"):
        ax.scatter(
            group["flesch_reading_ease"],
            group["perplexity"],
            label=model,
            alpha=0.7,
        )
    ax.set_xlabel("Flesch Reading Ease (higher = easier)")
    ax.set_ylabel("Perplexity (lower = more fluent)")
    ax.set_title("Readability vs fluency per paper")
    ax.legend()
    plt.tight_layout()
    plt.show()